# Hyb spot intensity QC — experiment-wide spot-intensity overview

Complements `fast_spot_quantification.ipynb`'s fixed-handful per-bit QC with a coarser, experiment-wide check: for `N_SELECTED_FOVS` FOVs evenly spaced across the whole tissue (fov_id 0 -> max, not a hand-picked list), measures each detected focus's background-subtracted peak intensity per (round, color) and plots the resulting intensity distribution, so a systematic drop in spot brightness (relative to the `FOCI_THRESH_SIGMA` detection threshold) can be caught while imaging is still in progress.

Reuses the exact same detection/caching machinery as `fast_spot_quantification.ipynb` (`MERci.analysis.fast_spot_quantification`): sample `N_Z_SAMPLES` z-planes per (round, color), read just those frames, max-project, then detect foci via `MERci.analysis.spot_localization.detect_beads_2d`. Unlike that notebook, frames are **not** cropped (`CROP_SIZE = None`) -- this notebook samples many FOVs spread across the whole tissue rather than repeatedly re-measuring the same few, so there is no single edge-vignetting profile worth cropping away; a full-frame max-projection just gives a coarser, whole-tissue signal instead.

Results are cached to `SAMPLE_DIR/analysis/cache/hyb_spot_intensity_qc/` as one CSV per (round, color, FOV) -- re-running the notebook (e.g. as more rounds/hybs are imaged) only computes new combinations; set `OVERWRITE_OUTPUT = True` to force recomputation.

Section 3 plots which FOVs were selected against the tissue boundary, as a sanity check that the evenly-spaced sample actually spans the imaged tissue. The final plot pools every `SELECTED_FOVS` FOV's foci together per round: one row per real bit color, x = round number, one semi-transparent violin per round showing that round's spot-intensity distribution for that color.

## 1 — Setup

In [ ]:
%matplotlib inline
# %matplotlib widget  # uncomment for interactive pan/zoom (ipympl) -- roughly
#                       doubles output size vs. inline: the widget's static-
#                       html fallback embeds a second copy of each rendered
#                       image alongside the PNG output

import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import read_image_frames
from MERci.progress_display       import ProgressReporter
from MERci.plots.experiment_plots          import get_merci_figures_dir, plot_fov_layout
from MERci.acquisition.positions  import (
    resolve_boundaries_source_dir, discover_boundary_files, load_boundary_polygon,
)
from MERci.analysis.fast_spot_quantification import (
    select_evenly_spaced_fovs, resolve_round_color_frame_indices, detect_foci_in_crop,
    spot_cache_path, compute_fov_round_color_spots, load_all_spots,
)

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "hyb_spot_intensity_qc"   # namespaces this notebook's cache + figures
MICROSCOPE    = "MF3"

# How many FOVs to sample, evenly spaced across the whole experiment footprint (fov_id 0 ->
# max, see select_evenly_spaced_fovs below) -- a coarse experiment-wide overview, unlike
# fast_spot_quantification.ipynb's fixed handful of hand-picked FOVs for fine per-bit QC.
N_SELECTED_FOVS = 10

# Full-frame max-intensity-projection, no center-crop. fast_spot_quantification.ipynb crops to
# CROP_SIZE=400 because it repeatedly re-measures the SAME few FOVs and wants that comparison
# vignetting-free; this notebook instead samples many FOVs spread across the whole tissue for a
# coarse overview, so there's no single representative vignetting profile worth cropping away.
CROP_SIZE = None

# How many z-planes to sample per (round, color), evenly spaced across that round's own real
# z-sweep for the color (see Section 4) -- these are max-projected before spot detection
# (Section 5), so a focus slightly out of the plane closest to its true z is still caught.
N_Z_SAMPLES = 5

# Colors to exclude from every round's resolution (Section 4) -- 405 nm is the cells/DAPI
# round's own channel (not a hybridization bit), 488 nm is HAL's bead/focus-lock reference
# channel (always at a fixed bead z, never real tissue signal). Real bit colors (typically
# 560/650/750) are picked up automatically from each round's own frame table.
EXCLUDED_COLORS = [405.0, 488.0]

# Foci-detection parameters for detect_beads_2d (Section 5) -- tuned by eye against the
# diagnostic overlay in Section 6, not calibrated against ground truth. FOCI_MIN_DIST_PX is the
# minimum center-to-center spacing (pixels) between accepted foci; FOCI_THRESH_SIGMA is the
# detection threshold above the estimated background, in units of the background's own std.
FOCI_MIN_DIST_PX  = 4
FOCI_THRESH_SIGMA = 3.0

# False (default): skip recomputing any (round, color, FOV) combination that already has a
# cached CSV in OUTPUT_DIR (Section 7) -- lets this notebook be re-run cheaply as new
# rounds/hybs are imaged. True: recompute and overwrite every combination regardless of what's
# already cached.
OVERWRITE_OUTPUT = False

# Shared plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- reused by every plotting cell in this
# notebook.
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 10
PLOT_LEGEND_FONTSIZE = 10

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# Evenly-spaced FOV sample across the whole experiment footprint (see this notebook's own intro
# cell / N_SELECTED_FOVS above).
SELECTED_FOVS = select_evenly_spaced_fovs(sorted(meta.fovs), N_SELECTED_FOVS)

# Own cache directory (NOTEBOOK_GUIDELINES.md #2), distinct from fast_spot_quantification
# .ipynb's own analysis/fast_spot_quantification/ -- different notebook name means the two
# caches never collide even though both key on round/color/fov.
OUTPUT_DIR = SAMPLE_DIR / "analysis" / "cache" / "hyb_spot_intensity_qc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "during_imaging", NOTEBOOK_NAME)

print(f"Sample name    : {SAMPLE_NAME}")
print(f"FOVs (total)   : {meta.n_fovs}")
print(f"Rounds         : {sorted(meta.rounds)}")
print(f"SELECTED_FOVS  : {SELECTED_FOVS}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")

## 3 — Plot: selected FOVs vs tissue boundary

Sanity check for `SELECTED_FOVS` (Section 2) -- shows which of the experiment's FOVs were picked for this coarse overview, plotted against the tissue boundary(ies) discovered under `positions/boundaries/{manual,from_mosaic}/` (same discovery convention as `02_create_positions_from_boundaries.ipynb`/`02_create_boundary_from_mosaic.ipynb`). If no boundary file exists yet, only the FOV scatter is shown.

In [ ]:
# Same discovery convention as 02_create_positions_from_boundaries.ipynb /
# 02_create_boundary_from_mosaic.ipynb (positions/boundaries/{manual,from_mosaic}/) -- reused
# here purely to display the outline for context, not to build any FOV grid. Skips (no error,
# just a note) if nothing has been drawn yet.
POSITIONS_DIR = SAMPLE_DIR / "positions"
BOUNDARY_POLYGONS = []
try:
    source_dir, boundary_source = resolve_boundaries_source_dir(POSITIONS_DIR)
    boundaries, _ = discover_boundary_files(source_dir)
    BOUNDARY_POLYGONS = [load_boundary_polygon(b.path) for b in boundaries]
    print(f"Loaded {len(BOUNDARY_POLYGONS)} boundary polygon(s) from boundaries/{boundary_source}/")
except FileNotFoundError:
    print("No tissue boundary file found yet under positions/boundaries/{manual,from_mosaic}/ -- "
          "showing FOV layout without a boundary overlay.")

fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.selected_fovs_vs_boundary.png"
plot_fov_layout(
    meta, highlight_fov_ids=SELECTED_FOVS,
    title=f"Selected FOVs for hyb-spot-intensity QC ({len(SELECTED_FOVS)} of {meta.n_fovs})",
    save_path=fig_path, boundary_polygons=BOUNDARY_POLYGONS,
)
print(f"Saved: {fig_path}")

## 4 — Resolve each round's real bit colors -> sampled z-frame indices

Mirrors `round_mosaics.ipynb`'s own frame-table resolution (Section 3 there), but instead of the single frame closest to a target z, keeps `N_Z_SAMPLES` frame indices evenly spaced across `get_all_color_frame_indices`' full ascending-z list for that color -- these get max-projected before spot detection (Section 5).

In [ ]:
ROUND_COLOR_FRAME_INDICES = {}
for round_id in sorted(meta.rounds):
    cf = resolve_round_color_frame_indices(round_id, config, meta, EXCLUDED_COLORS, N_Z_SAMPLES)
    if cf:
        ROUND_COLOR_FRAME_INDICES[round_id] = cf
        n_samples_per_color = {c: len(f) for c, f in cf.items()}
        print(f"Round {round_id}: colors {sorted(cf)} -> n_z_samples per color: {n_samples_per_color}")
    else:
        print(f"Round {round_id}: no bit colors resolved")

## 5 — Foci detection

Foci detection (`detect_foci_in_crop`, `MERci.analysis.fast_spot_quantification`) crops every sampled z-frame to `CROP_SIZE` -- here `None`, so it is left as the full frame (see `crop_center`) -- max-projects, estimates a background median, then detects candidate foci via `detect_beads_2d`. Used by both the diagnostic overlay (Section 6) and the real per-combination computation (Section 7) below.

## 6 — Diagnostic: visualize detected foci on one example FOV

Before running the full batch below, sanity-check `FOCI_MIN_DIST_PX`/`FOCI_THRESH_SIGMA` (Section 2) against one real, already-imaged (round, color, FOV) combination -- re-run this cell after adjusting either parameter. Picks the first resolved combination that has at least one `SELECTED_FOVS` entry already imaged; prints a message instead if nothing is imaged yet. Shows the full-frame max-projection (no crop, since `CROP_SIZE = None`).

In [ ]:
EXAMPLE = None
for round_id, color_frames in ROUND_COLOR_FRAME_INDICES.items():
    series = meta.series_for_round(round_id)
    for color_nm, frame_indices in color_frames.items():
        for fov_id in SELECTED_FOVS:
            existing = [p for p in (s.resolve_path(fov_id, config.image_suffix) for s in series) if p.exists()]
            if existing:
                EXAMPLE = (round_id, color_nm, fov_id, frame_indices, existing[0])
                break
        if EXAMPLE:
            break
    if EXAMPLE:
        break

if EXAMPLE is None:
    print("No imaged (round, color, FOV) combination found yet for the diagnostic overlay -- "
          "run again once imaging has started.")
else:
    example_round_id, example_color_nm, example_fov_id, example_frame_indices, example_path = EXAMPLE
    frames = read_image_frames(example_path, example_frame_indices,
                                frame_width=config.frame_width, frame_height=config.frame_height)
    max_proj, bg_med, candidates = detect_foci_in_crop(frames, CROP_SIZE, FOCI_MIN_DIST_PX, FOCI_THRESH_SIGMA)

    vmin, vmax = np.percentile(max_proj, [1.0, 99.0])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(max_proj, cmap="gray", vmin=vmin, vmax=vmax)
    if len(candidates):
        ax.scatter(candidates[:, 1], candidates[:, 0], s=40, facecolors="none",
                   edgecolors="red", linewidths=1.0)
    ax.set_title(f"round {example_round_id}, {example_color_nm:.0f} nm, FOV {example_fov_id} "
                 f"-- {len(candidates)} foci detected", fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
    fig.tight_layout()
    fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.example_foci_overlay.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved: {fig_path}")

## 7 — Run the quantification

For every (round, color, FOV) combination resolved above, skips any that already has a cached CSV in `OUTPUT_DIR` unless `OVERWRITE_OUTPUT=True` -- so re-running this notebook as more rounds/hybs get imaged only computes what's new. A combination whose FOV isn't imaged yet is skipped without caching anything, so it's retried on a later run rather than permanently recorded as empty.

In [ ]:
combos = [
    (fov_id, round_id, color_nm, frame_indices)
    for round_id, color_frames in ROUND_COLOR_FRAME_INDICES.items()
    for color_nm, frame_indices in color_frames.items()
    for fov_id in SELECTED_FOVS
]

n_computed, n_skipped_cached, n_not_imaged = 0, 0, 0
reporter = ProgressReporter(total=len(combos), label="Quantifying foci")
for fov_id, round_id, color_nm, frame_indices in reporter.wrap(combos):
    cache_path = spot_cache_path(OUTPUT_DIR, round_id, color_nm, fov_id)
    if cache_path.exists() and not OVERWRITE_OUTPUT:
        n_skipped_cached += 1
        continue

    series = meta.series_for_round(round_id)
    df = compute_fov_round_color_spots(fov_id, round_id, color_nm, frame_indices, series,
                                        config, CROP_SIZE, FOCI_MIN_DIST_PX, FOCI_THRESH_SIGMA)
    if df is None:
        n_not_imaged += 1
        continue

    df.to_csv(cache_path, index=False)
    n_computed += 1

print(f"Computed: {n_computed} | Skipped (already cached): {n_skipped_cached} | Not yet imaged: {n_not_imaged}")

## 8 — Load cached results

In [ ]:
SPOTS_DF = load_all_spots(OUTPUT_DIR, ROUND_COLOR_FRAME_INDICES, SELECTED_FOVS)
n_combos = SPOTS_DF[["fov", "round", "color_nm"]].drop_duplicates().shape[0] if not SPOTS_DF.empty else 0
print(f"Loaded {len(SPOTS_DF)} foci across {n_combos} (fov, round, color) combination(s).")

## 9 — Plot: spot-intensity violin per bit color, across rounds

One row per real bit color found in `SPOTS_DF`; within each row, one semi-transparent (`alpha=0.5`) violin per round, positioned at that round's actual `imaging_round` number on the x-axis, pooling every `SELECTED_FOVS` FOV's foci together. A round whose reagent degraded should show up as a visibly lower and/or tighter violin relative to its neighbours.

In [ ]:
if SPOTS_DF.empty:
    print("No foci data yet -- nothing to plot. Run Section 7 once at least one combination is imaged.")
else:
    colors_present = sorted(SPOTS_DF["color_nm"].unique())
    fig, axes = plt.subplots(len(colors_present), 1, figsize=(10, 4 * len(colors_present)), squeeze=False)

    for ax, color_nm in zip(axes[:, 0], colors_present):
        sub = SPOTS_DF[SPOTS_DF["color_nm"] == color_nm]
        rounds_present = sorted(sub["round"].unique())
        per_round_data = [sub.loc[sub["round"] == r, "intensity"].values for r in rounds_present]
        # violinplot errors on an empty array -- drop rounds with zero detected
        # foci rather than crashing the whole subplot.
        valid = [(r, d) for r, d in zip(rounds_present, per_round_data) if len(d) > 0]
        if valid:
            vp_rounds, vp_data = zip(*valid)
            parts = ax.violinplot(vp_data, positions=vp_rounds, showmedians=True)
            for body in parts["bodies"]:
                body.set_alpha(0.5)
            ax.set_xticks(vp_rounds)
        ax.set_title(f"{color_nm:.0f} nm", fontsize=PLOT_TITLE_FONTSIZE)
        ax.set_xlabel("Round", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_ylabel("Spot intensity (bg-subtracted)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

    fig.tight_layout()
    fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.spot_intensity_violin.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved: {fig_path}")